In [9]:
import json
import numpy as np
import pandas as pd
import psycopg2
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# ======================
# CONFIG
# ======================
TOP_K = 10
LABEL_FILE = "labels.json"

DB_CONFIG = {
    "host": "aws-1-ap-northeast-2.pooler.supabase.com",
    "port": 6543,
    "database": "postgres",
    "user": "postgres.ypkwqsbsunlvpoqdadbq",
    "password": "5$eAK8EV4S+gsKj",
    "sslmode": "require"
}

# ======================
# LOAD LABELS
# ======================
def load_labels(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    gt = defaultdict(dict)

    for item in data:
        gt[item["query"]][item["restaurant_id"]] = item["relevance"]

    return gt

# ======================
# LOAD RESTAURANTS FROM DB
# ======================
def load_restaurants():

    conn = psycopg2.connect(**DB_CONFIG)

    query = """
        SELECT 
    r.id,
    r.name,
    r.name || ' ' || re.content AS text,
    re.embedding
FROM restaurants r
JOIN restaurant_embeddings re 
ON r.id = re.restaurant_id
    """

    df = pd.read_sql(query, conn)

    # convert embedding string → numpy
    df["embedding"] = df["embedding"].apply(
        lambda x: np.array(json.loads(x))
    )

    return df

# ======================
# TF-IDF
# ======================
def build_tfidf(df):
    vectorizer = TfidfVectorizer(
        max_features=5000,
        lowercase=True,
        ngram_range=(1,2)
    )
    tfidf_matrix = vectorizer.fit_transform(df["text"])
    return vectorizer, tfidf_matrix

# ======================
# MODEL (chỉ encode query)
# ======================
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# ======================
# METRICS
# ======================
def precision_at_k(ranked_ids, gt_dict, k):
    top_k = ranked_ids[:k]
    if not top_k:
        return 0.0
    hits = sum(1 for rid in top_k if gt_dict.get(rid, 0) > 0)
    return hits / len(top_k)


def recall_at_k(ranked_ids, gt_dict, k):
    relevant_items = [rid for rid, rel in gt_dict.items() if rel > 0]
    if not relevant_items:
        return 0.0
    top_k = ranked_ids[:k]
    hits = sum(1 for rid in top_k if gt_dict.get(rid, 0) > 0)
    return hits / len(relevant_items)


def ndcg_at_k(ranked_ids, gt_dict, k):
    dcg = sum(
        (2**gt_dict.get(rid, 0) - 1) / np.log2(i + 2)
        for i, rid in enumerate(ranked_ids[:k])
    )

    ideal_rels = sorted(gt_dict.values(), reverse=True)[:k]
    idcg = sum(
        (2**rel - 1) / np.log2(i + 2)
        for i, rel in enumerate(ideal_rels)
    )

    return dcg / idcg if idcg > 0 else 0.0

# ======================
# RETRIEVAL
# ======================
def retrieve_tfidf(query, vectorizer, tfidf_matrix, df):

    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, tfidf_matrix)[0]

    ranked_ids = df.iloc[np.argsort(-scores)]["id"].tolist()
    return ranked_ids


def retrieve_milm(query, df):

    query_emb = model.encode(query)

    emb_matrix = np.stack(df["embedding"].values)

    scores = cosine_similarity([query_emb], emb_matrix)[0]

    ranked_ids = df.iloc[np.argsort(-scores)]["id"].tolist()
    return ranked_ids

# ======================
# EVALUATION
# ======================
def evaluate():

    print("📥 Loading data...")
    gt = load_labels(LABEL_FILE)
    df = load_restaurants()

    print("🔧 Building TF-IDF...")
    vectorizer, tfidf_matrix = build_tfidf(df)

    print("✅ Using precomputed embeddings from DB")

    results = {
        "tfidf": {"precision": [], "recall": [], "ndcg": []},
        "milm": {"precision": [], "recall": [], "ndcg": []}
    }

    for query, gt_dict in gt.items():

        tfidf_rank = retrieve_tfidf(query, vectorizer, tfidf_matrix, df)
        milm_rank = retrieve_milm(query, df)

        # TF-IDF
        results["tfidf"]["precision"].append(
            precision_at_k(tfidf_rank, gt_dict, TOP_K)
        )
        results["tfidf"]["recall"].append(
            recall_at_k(tfidf_rank, gt_dict, TOP_K)
        )
        results["tfidf"]["ndcg"].append(
            ndcg_at_k(tfidf_rank, gt_dict, TOP_K)
        )

        # MiniLM
        results["milm"]["precision"].append(
            precision_at_k(milm_rank, gt_dict, TOP_K)
        )
        results["milm"]["recall"].append(
            recall_at_k(milm_rank, gt_dict, TOP_K)
        )
        results["milm"]["ndcg"].append(
            ndcg_at_k(milm_rank, gt_dict, TOP_K)
        )

    # ======================
    # FINAL RESULT
    # ======================
    print("\n===== FINAL RESULT =====")

    for model_name in ["tfidf", "milm"]:
        print(f"\n🔹 {model_name.upper()}")
        print("Precision@K:", round(np.mean(results[model_name]["precision"]), 4))
        print("Recall@K:", round(np.mean(results[model_name]["recall"]), 4))
        print("NDCG@K:", round(np.mean(results[model_name]["ndcg"]), 4))


if __name__ == "__main__":
    evaluate()

📥 Loading data...


C:\Users\Admin\AppData\Local\Temp\ipykernel_17688\2061211728.py:57: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


🔧 Building TF-IDF...
✅ Using precomputed embeddings from DB

===== FINAL RESULT =====

🔹 TFIDF
Precision@K: 0.04
Recall@K: 0.035
NDCG@K: 0.1342

🔹 MILM
Precision@K: 0.36
Recall@K: 0.3965
NDCG@K: 0.3269
